In [2]:
import numpy as np
import gc
from matplotlib import pyplot as plt
import time 

import sys
sys.path.append('/dipc/kstoreyf/muchisimocks/scripts')
import compute_statistics as cs
import data_loader
import utils
import plotter

import bacco

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Load tracer field

In [3]:
tag_params = '_shame_p0_n1000'
box_size = 1000.0
dir_mocks = f'/scratch/kstoreyf/muchisimocks/muchisimocks_lib{tag_params}'
idx_mock = 0
fn_fields = f'{dir_mocks}/mock{idx_mock}/bias_fields_eul_deconvolved_{idx_mock}.npy'
bias_terms_eul = np.load(fn_fields)

In [4]:
params_df, param_dict_fixed = data_loader.load_cosmo_params(tag_params)
if params_df is None:
    param_dict = param_dict_fixed
else:
    param_dict = params_df.iloc[0].to_dict()
    param_dict.update(param_dict_fixed)
cosmo = utils.get_cosmo(param_dict)

In [5]:
tag_mock = '_nbar0.00022'
param_dict_forbias = data_loader.load_params_ood('shame', tag_mock)
bias_vector = [param_dict_forbias[name] for name in utils.biasparam_names_ordered]

In [6]:
n_grid_orig = 512
bs = np.concatenate(([1.0], bias_vector))
tracer_field_det = utils.get_tracer_field(bias_terms_eul, bias_vector, n_grid_orig, 
                                             noise_field=None, A_noise=None)

# Power spectra

In [ ]:
k_min = 0.01
k_max = 0.1
n_bins = 3



In [ ]:
n_grid = tracer_field_det.shape[-1]
args_power_grid = {
    "ngrid": n_grid,
    "box": box_size,
    "interlacing": False, #default: True
    "deposit_method": 'cic', #default: "tsc",
    "log_binning": True,
    "kmin": k_min,
    "kmax": k_max,
    "nbins": n_bins,
    "cosmology": cosmo,
}

In [17]:
pk_obj = bacco.statistics.compute_crossspectrum_twogrids(
                    grid1=tracer_field_det,
                    grid2=tracer_field_det,
                    **args_power_grid)

2026-02-26 08:56:28,475 bacco.statistics : Computing the power spectrum with ngrid=128 and interlacing=False
2026-02-26 08:56:28,522 bacco.statistics :  ...done in 0.0471 s


bacco.power : boxsize 1000.000000; ngrid 128; nthreads 1; interlacing 0; deposit_method 1; log_binning 1; type 1; precision=single; correct_grid=0 (log=1); correct_sn=0
bacco.power : normalise_grid1=0 normalise_grid2=0 deconvolve_grid1=0 deconvolve_grid2=0
bacco.power : total mass 0.999999 (grid1) 0.999999 (grid2)
bacco.power : Doing FFTW
bacco.power: FFT took 0.014025 sec
bacco.power : Counting modes
bacco.power : done counting modes in 0.020433 sec
bacco.power : Starting Fourier loop 
bacco.power : done Fourier loop in 0.000012 secs
bacco.power : Deallocating arrays


In [52]:
r_edges = np.logspace(np.log10(k_min), np.log10(k_max), n_bins+1)
r_centers = np.sqrt(r_edges[:-1] * r_edges[1:])
print('Expected r_edges:', r_edges)
print('Expected r_centers:', r_centers)

dlog = np.log10(k_max/k_min)/(n_bins-1)
r_edges_match = np.logspace(np.log10(k_min), np.log10(k_max) + dlog, n_bins+1)
print('Matched r_edges:', r_edges_match)
r_centers_match = np.sqrt(r_edges_match[:-1] * r_edges_match[1:])
print('Matched r_centers:', r_centers_match)
print('pk_obj["k"]:', pk_obj['k'])

Expected r_edges: [0.1        0.21544347 0.46415888 1.        ]
Expected r_centers: [0.14677993 0.31622777 0.68129207]
Matched r_edges: [0.1        0.31622777 1.         3.16227766]
Matched r_centers: [0.17782794 0.56234133 1.77827941]
pk_obj["k"]: [0.01778279 0.05623413 0.17782795]


In [36]:
k_min, k_max, n_bins = 0.1, 1.0, 3

K0 = k_min
K1 = k_max
bins_ps = n_bins

binfac = (bins_ps - 1) / np.log(K1 / K0)
print(binfac)

edges = [K0 * np.exp(i / binfac) for i in range(bins_ps + 1)]
centers = [np.exp((i + 0.5) / binfac + np.log(K0)) for i in range(bins_ps)]

print("edges:  ", edges)
print("centers:", centers)

0.8685889638065037
edges:   [0.1, 0.31622776601683794, 0.9999999999999999, 3.162277660168378]
centers: [0.1778279410038923, 0.5623413251903491, 1.7782794100389225]


In [53]:
k_min, k_max, n_bins = 0.1, 1.0, 3

K0 = k_min
K1 = k_max
bins_ps = n_bins

binfac_corr = (bins_ps) / np.log(K1 / K0)
print(binfac)

edges = [K0 * np.exp(i / binfac_corr) for i in range(bins_ps + 1)]
centers = [np.exp((i + 0.5) / binfac_corr + np.log(K0)) for i in range(bins_ps)]

print("edges:  ", edges)
print("centers:", centers)

0.8685889638065037
edges:   [0.1, 0.21544346900318834, 0.46415888336127786, 0.9999999999999999]
centers: [0.14677992676220697, 0.31622776601683794, 0.6812920690579614]


In [37]:
edges = [K0 * np.exp(i / binfac) for i in range(bins_ps + 1)]
centers = [np.exp((i + 0.5) / bins_ps + np.log(K0)) for i in range(bins_ps)]

print("edges:  ", edges)
print("centers:", centers)

edges:   [0.1, 0.31622776601683794, 0.9999999999999999, 3.162277660168378]
centers: [0.1181360412865646, 0.16487212707001286, 0.23009758908928257]


## Linear

In [38]:
n_grid = tracer_field_det.shape[-1]
args_power_grid_lin = {
    "ngrid": n_grid,
    "box": box_size,
    "interlacing": False, #default: True
    "deposit_method": 'cic', #default: "tsc",
    "log_binning": False,
    "kmin": k_min,
    "kmax": k_max,
    "nbins": n_bins,
    "cosmology": cosmo,
}
pk_obj_lin = bacco.statistics.compute_crossspectrum_twogrids(
                    grid1=tracer_field_det,
                    grid2=tracer_field_det,
                    **args_power_grid_lin)

2026-02-26 09:25:50,245 bacco.statistics : Computing the power spectrum with ngrid=128 and interlacing=False


bacco.power : boxsize 1000.000000; ngrid 128; nthreads 64; interlacing 0; deposit_method 1; log_binning 0; type 1; precision=single; correct_grid=1 (log=1); correct_sn=0
bacco.power : normalise_grid1=0 normalise_grid2=0 deconvolve_grid1=0 deconvolve_grid2=0
bacco.power : total mass 0.999999 (grid1) 0.999999 (grid2)
bacco.power : Doing FFTW
bacco.power: FFT took 0.066831 sec
bacco.power : Counting modes
bacco.power : done counting modes in 0.209135 sec
bacco.power : Starting Fourier loop 
bacco.power : Doing inverse FFT
bacco.power : Creating final array
bacco.power : done Fourier loop in 0.491245 secs
bacco.power : Deallocating arrays


2026-02-26 09:25:51,234 bacco.util : pk multipoles at k 0.8500000002483528 set to zero: it seems you have a lot of bins for this grid size
2026-02-26 09:25:51,235 bacco.statistics :  ...done in 0.99 s


In [43]:
r_edges_lin = np.linspace(k_min, k_max, n_bins+1)
r_centers_lin = (r_edges_lin[:-1] + r_edges_lin[1:]) / 2
print("Expected r_edges_lin:", r_edges_lin)
print("Expected r_centers_lin:", r_centers_lin)

print("pk_obj_lin['k']:", pk_obj_lin['k'])

Expected r_edges_lin: [0.1 0.4 0.7 1. ]
Expected r_centers_lin: [0.25 0.55 0.85]
pk_obj_lin['k']: [0.25 0.55 0.85]
